# NKAI Studio — Colab Voice + Talking Avatar

Runs without Docker. Uses Kokoro for voice generation and MuseTalk 1.5 for portrait/video lip-sync. This is a first-stage **rendered talking-face** workflow, not yet a full-body avatar or real-time streaming API. Keep the existing Qwen worker unchanged.

In [ ]:
import os, shutil, subprocess, sys
import torch
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('Select Runtime > Change runtime type > GPU, then rerun this cell.')
print('Free disk (GiB):', round(shutil.disk_usage('/content').free / 1024**3, 2))


## 1. Install the voice engine

Kokoro-82M has Apache-2.0 weights. This installs into the current Colab Python environment; it does not touch your Qwen server process. If the existing notebook kernel has unusual package overrides, restart only if you are prepared to rerun the current runtime setup.

In [ ]:
!apt-get -qq update && apt-get -qq install -y espeak-ng ffmpeg
!pip -q install 'kokoro>=0.9.2' soundfile
import importlib.metadata
print('kokoro:', importlib.metadata.version('kokoro'))
print('GPU still available:', torch.cuda.is_available())


In [ ]:
from kokoro import KPipeline
import soundfile as sf
import numpy as np

VOICE_TEXT = 'Hello, welcome to NKAI Studio. I am your AI career companion. How can I help you today?'
VOICE_ID = 'af_heart'  # Kokoro English voice; choose another supported voice if desired.
kokoro = KPipeline(lang_code='a', device='cuda' if torch.cuda.is_available() else 'cpu')
chunks = []
for _, _, audio in kokoro(VOICE_TEXT, voice=VOICE_ID, speed=1.0):
    chunks.append(audio.detach().cpu().numpy() if hasattr(audio, 'detach') else np.asarray(audio))
if not chunks:
    raise RuntimeError('Kokoro returned no audio chunks.')
audio = np.concatenate(chunks).astype('float32')
VOICE_WAV = '/content/nkai_voice.wav'
sf.write(VOICE_WAV, audio, 24000)
print('Saved:', VOICE_WAV, '| seconds:', round(len(audio)/24000, 2), '| samples:', len(audio))
from IPython.display import Audio, display
display(Audio(VOICE_WAV))


## 2. Prepare MuseTalk in a separate Python 3.10 environment

MuseTalk's upstream setup targets Python 3.10 and PyTorch 2.0.1. A separate environment avoids downgrading the Colab kernel's PyTorch. This install can take time and several GB of disk. If any dependency fails, stop and share the full error rather than changing the working Qwen environment.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = '/root/.local/bin:' + os.environ.get('PATH', '')
!uv python install 3.10
!git clone https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk
!uv venv --python 3.10 /content/musetalk-venv
!uv pip install --python /content/musetalk-venv/bin/python torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!uv pip install --python /content/musetalk-venv/bin/python -r /content/MuseTalk/requirements.txt
!uv pip install --python /content/musetalk-venv/bin/python openmim
!/content/musetalk-venv/bin/mim install mmengine
!/content/musetalk-venv/bin/mim install mmcv==2.0.1
!/content/musetalk-venv/bin/mim install mmdet==3.1.0
!/content/musetalk-venv/bin/mim install mmpose==1.1.0
!ffmpeg -version | head -n 1


## 3. Download the avatar/lip-sync model weights

This downloads the upstream MuseTalk weights and supporting models. It may be large. Check free disk before running. Do not redistribute the weights without reviewing each model's license.

In [ ]:
free_gib = shutil.disk_usage('/content').free / 1024**3
print(f'Free disk before model download: {free_gib:.2f} GiB')
if free_gib < 12:
    raise RuntimeError('Less than 12 GiB free. Free space before downloading MuseTalk weights.')
%cd /content/MuseTalk
!bash download_weights.sh
required = ['/content/MuseTalk/models/musetalkV15/unet.pth', '/content/MuseTalk/models/musetalkV15/musetalk.json']
for p in required:
    print(p, os.path.exists(p), (os.path.getsize(p) if os.path.exists(p) else 0))
if not all(os.path.exists(p) for p in required):
    raise RuntimeError('MuseTalk v1.5 weights are incomplete; inspect download output before continuing.')


## 4. Upload a portrait or source video and render the talking avatar

Use an image/video you own or have permission to animate. MuseTalk generates a lip-synced video from the selected face and the Kokoro audio. It is not a full-body 3D avatar.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload one portrait image or a short source video.
if not uploaded:
    raise RuntimeError('Upload a portrait image or source video first.')
source_name = next(iter(uploaded))
SOURCE_MEDIA = '/content/' + source_name
print('Source media:', SOURCE_MEDIA)
print('Voice WAV:', VOICE_WAV)


In [ ]:
import yaml
from pathlib import Path
repo = Path('/content/MuseTalk')
cfg_path = repo / 'configs/inference/test.yaml'
cfg = yaml.safe_load(cfg_path.read_text())
cfg['video_path'] = SOURCE_MEDIA
cfg['audio_path'] = VOICE_WAV
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print('Inference config:', cfg_path)
print(cfg_path.read_text())
os.chdir(str(repo))
cmd = [
    '/content/musetalk-venv/bin/python', '-m', 'scripts.inference',
    '--inference_config', str(cfg_path),
    '--result_dir', '/content/nkai-avatar-results',
    '--unet_model_path', 'models/musetalkV15/unet.pth',
    '--unet_config', 'models/musetalkV15/musetalk.json',
    '--version', 'v15',
    '--ffmpeg_path', '/usr/bin/ffmpeg'
]
print('Running MuseTalk inference...')
completed = subprocess.run(cmd, text=True, capture_output=True)
print((completed.stdout or '')[-8000:])
print((completed.stderr or '')[-8000:])
if completed.returncode != 0:
    raise RuntimeError(f'MuseTalk inference failed with exit code {completed.returncode}. Share the logs above.')
print('Results:', list(Path('/content/nkai-avatar-results').glob('*')))


In [ ]:
from pathlib import Path
from IPython.display import Video, display
results = [p for p in Path('/content/nkai-avatar-results').rglob('*') if p.suffix.lower() in ('.mp4', '.mov')]
if not results:
    print('No video found. Inspect the inference output above; MuseTalk output paths can vary by version.')
else:
    final_video = max(results, key=lambda p: p.stat().st_mtime)
    print('Final video:', final_video)
    display(Video(str(final_video), embed=True))
    files.download(str(final_video))
